In [31]:
from pathlib import Path

import numpy as np
import tensorflow as tf

from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

In [32]:
PROJECT_DIR = Path(r"D:\proto\TextileDefectPrototype")

NORMAL_DIR = PROJECT_DIR / "dataset" / "normal"
DEFECT_DIR = PROJECT_DIR / "dataset" / "defect"

MODEL_DIR = PROJECT_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODEL_DIR / "mobilenetv2_classifier.keras"

print("Project:", PROJECT_DIR)
print("Normal folder:", NORMAL_DIR)
print("Defect folder:", DEFECT_DIR)
print("Model path:", MODEL_PATH)

Project: D:\proto\TextileDefectPrototype
Normal folder: D:\proto\TextileDefectPrototype\dataset\normal
Defect folder: D:\proto\TextileDefectPrototype\dataset\defect
Model path: D:\proto\TextileDefectPrototype\models\mobilenetv2_classifier.keras


In [33]:
print("Normal folder exists:", NORMAL_DIR.exists())
print("Defect folder exists:", DEFECT_DIR.exists())
print("Models folder exists:", MODEL_DIR.exists())

Normal folder exists: True
Defect folder exists: True
Models folder exists: True


In [34]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg"}

normal_files = [
    str(p)
    for p in NORMAL_DIR.iterdir()
    if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
]

defect_files = [
    str(p)
    for p in DEFECT_DIR.iterdir()
    if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
]

print("Normal JPG/JPEG images:", len(normal_files))
print("Defect JPG/JPEG images:", len(defect_files))
print("Total images:", len(normal_files) + len(defect_files))

Normal JPG/JPEG images: 5794
Defect JPG/JPEG images: 590
Total images: 6384


In [35]:
file_paths = normal_files + defect_files

labels = (
    [0] * len(normal_files)
    + [1] * len(defect_files)
)

file_paths = np.array(file_paths)
labels = np.array(labels)

print("Total paths:", len(file_paths))
print("Normal labels:", np.sum(labels == 0))
print("Defect labels:", np.sum(labels == 1))

Total paths: 6384
Normal labels: 5794
Defect labels: 590


In [36]:
print("First 5 normal images:")

for path in normal_files[:5]:
    print(path)

print("\nFirst 5 defect images:")

for path in defect_files[:5]:
    print(path)

First 5 normal images:
D:\proto\TextileDefectPrototype\dataset\normal\T1-D0-001.jpg
D:\proto\TextileDefectPrototype\dataset\normal\T1-D0-002.jpg
D:\proto\TextileDefectPrototype\dataset\normal\T1-D0-003.jpg
D:\proto\TextileDefectPrototype\dataset\normal\T1-D0-004.jpg
D:\proto\TextileDefectPrototype\dataset\normal\T1-D0-005.jpg

First 5 defect images:
D:\proto\TextileDefectPrototype\dataset\defect\T1-D12-029.jpg
D:\proto\TextileDefectPrototype\dataset\defect\T1-D12-030.jpg
D:\proto\TextileDefectPrototype\dataset\defect\T1-D12-031.jpg
D:\proto\TextileDefectPrototype\dataset\defect\T1-D12-032.jpg
D:\proto\TextileDefectPrototype\dataset\defect\T1-D12-033.jpg


In [37]:
train_paths, val_paths, train_labels, val_labels = train_test_split(
    file_paths,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

print("Training images:", len(train_paths))
print("Validation images:", len(val_paths))

Training images: 5107
Validation images: 1277


In [38]:
print("Training normal:", np.sum(train_labels == 0))
print("Training defect:", np.sum(train_labels == 1))

print("Validation normal:", np.sum(val_labels == 0))
print("Validation defect:", np.sum(val_labels == 1))

Training normal: 4635
Training defect: 472
Validation normal: 1159
Validation defect: 118


In [39]:
classes = np.unique(train_labels)

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_labels
)

class_weights = {
    int(cls): float(weight)
    for cls, weight in zip(classes, weights)
}

print("Class weights:", class_weights)

Class weights: {0: 0.5509169363538295, 1: 5.4099576271186445}


In [40]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

print("Image size:", IMG_SIZE)
print("Batch size:", BATCH_SIZE)

Image size: (224, 224)
Batch size: 32


In [41]:
def load_image(path, label):
    image = tf.io.read_file(path)

    image = tf.image.decode_image(
        image,
        channels=3,
        expand_animations=False
    )

    image = tf.image.resize(image, IMG_SIZE)

    image = tf.cast(image, tf.float32)

    image = preprocess_input(image)

    label = tf.cast(label, tf.float32)

    return image, label

In [42]:
train_ds = tf.data.Dataset.from_tensor_slices(
    (train_paths, train_labels)
)

val_ds = tf.data.Dataset.from_tensor_slices(
    (val_paths, val_labels)
)

In [43]:
train_ds = (
    train_ds
    .shuffle(2000, seed=42)
    .map(
        load_image,
        num_parallel_calls=tf.data.AUTOTUNE
    )
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds = (
    val_ds
    .map(
        load_image,
        num_parallel_calls=tf.data.AUTOTUNE
    )
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

In [45]:
for images, labels_batch in train_ds.take(1):
    print("Image batch shape:", images.shape)
    print("Label batch shape:", labels_batch.shape)

Image batch shape: (32, 224, 224, 3)
Label batch shape: (32,)


In [46]:
base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False

print("MobileNetV2 loaded.")

MobileNetV2 loaded.


In [47]:
model = models.Sequential([
    layers.Input(shape=(224, 224, 3)),

    base_model,

    layers.GlobalAveragePooling2D(),

    layers.Dense(
        128,
        activation="relu"
    ),

    layers.Dropout(0.3),

    layers.Dense(
        1,
        activation="sigmoid"
    )
])

In [48]:
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,081 (9.24 MB)

 Trainable params: 164,097 (641.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [49]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-4
    ),

    loss="binary_crossentropy",

    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall")
    ]
)

In [50]:
EPOCHS = 3

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    class_weight=class_weights
)

Epoch 1/3


d:\proto\TextileDefectPrototype\.venv\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


160/160 ━━━━━━━━━━━━━━━━━━━━ 165s 996ms/step - accuracy: 0.7601 - loss: 0.5317 - precision: 0.2246 - recall: 0.6504 - val_accuracy: 0.9099 - val_loss: 0.3517 - val_precision: 0.5091 - val_recall: 0.7119
Epoch 2/3
160/160 ━━━━━━━━━━━━━━━━━━━━ 153s 952ms/step - accuracy: 0.8845 - loss: 0.3584 - precision: 0.4320 - recall: 0.7945 - val_accuracy: 0.9287 - val_loss: 0.3116 - val_precision: 0.5860 - val_recall: 0.7797
Epoch 3/3
160/160 ━━━━━━━━━━━━━━━━━━━━ 153s 952ms/step - accuracy: 0.9031 - loss: 0.3109 - precision: 0.4860 - recall: 0.8453 - val_accuracy: 0.9499 - val_loss: 0.2371 - val_precision: 0.7177 - val_recall: 0.7542


In [51]:
results = model.evaluate(
    val_ds,
    verbose=1
)

print("\nEvaluation results:")
for name, value in zip(model.metrics_names, results):
    print(f"{name}: {value:.4f}")

40/40 ━━━━━━━━━━━━━━━━━━━━ 31s 761ms/step - accuracy: 0.9499 - loss: 0.2371 - precision: 0.7177 - recall: 0.7542

Evaluation results:
loss: 0.2371
compile_metrics: 0.9499


In [52]:
model.save(MODEL_PATH)

print("Model saved successfully.")
print(MODEL_PATH)

Model saved successfully.
D:\proto\TextileDefectPrototype\models\mobilenetv2_classifier.keras


In [54]:
def predict_single_image(image_path):
    image = tf.io.read_file(str(image_path))

    image = tf.image.decode_image(
        image,
        channels=3,
        expand_animations=False
    )

    image = tf.image.resize(image, IMG_SIZE)

    image = tf.cast(image, tf.float32)

    image = preprocess_input(image)

    image = tf.expand_dims(image, axis=0)

    prediction = float(
        model.predict(image, verbose=0)[0][0]
    )

    if prediction >= 0.5:
        predicted_class = "DEFECT"
        confidence = prediction * 100
    else:
        predicted_class = "NORMAL"
        confidence = (1 - prediction) * 100

    print("Prediction:", predicted_class)
    print("Confidence:", f"{confidence:.2f}%")

    return predicted_class, confidence

In [55]:
test_path = NORMAL_DIR / "T1-D0-001.jpg"

print("Testing:", test_path)

predict_single_image(test_path)

Testing: D:\proto\TextileDefectPrototype\dataset\normal\T1-D0-001.jpg
Prediction: NORMAL
Confidence: 72.68%


('NORMAL', 72.68085479736328)

In [56]:
test_defect_path = DEFECT_DIR / "T1-D2-004.jpg"

print("Testing:", test_defect_path)

predict_single_image(test_defect_path)

Testing: D:\proto\TextileDefectPrototype\dataset\defect\T1-D2-004.jpg
Prediction: NORMAL
Confidence: 59.11%


('NORMAL', 59.110939502716064)

In [57]:
FAST_TRAIN_BATCHES = 80
FAST_VAL_BATCHES = 20

train_ds_fast = train_ds.take(FAST_TRAIN_BATCHES)
val_ds_fast = val_ds.take(FAST_VAL_BATCHES)

print("Fast training batches:", FAST_TRAIN_BATCHES)
print("Fast validation batches:", FAST_VAL_BATCHES)

Fast training batches: 80
Fast validation batches: 20


In [58]:
history = model.fit(
    train_ds_fast,
    validation_data=val_ds_fast,
    epochs=3,
    class_weight=class_weights
)

Epoch 1/3
80/80 ━━━━━━━━━━━━━━━━━━━━ 74s 917ms/step - accuracy: 0.9176 - loss: 0.2715 - precision: 0.5395 - recall: 0.8506 - val_accuracy: 0.9391 - val_loss: 0.2336 - val_precision: 0.6557 - val_recall: 0.6897
Epoch 2/3
80/80 ━━━━━━━━━━━━━━━━━━━━ 83s 935ms/step - accuracy: 0.9133 - loss: 0.2663 - precision: 0.5230 - recall: 0.8542 - val_accuracy: 0.9438 - val_loss: 0.2197 - val_precision: 0.6618 - val_recall: 0.7759
Epoch 3/3
80/80 ━━━━━━━━━━━━━━━━━━━━ 75s 938ms/step - accuracy: 0.9133 - loss: 0.2688 - precision: 0.5236 - recall: 0.8333 - val_accuracy: 0.8641 - val_loss: 0.3020 - val_precision: 0.3858 - val_recall: 0.8448


In [59]:
predict_single_image(
    DEFECT_DIR / "T1-D2-004.jpg"
)

Prediction: DEFECT
Confidence: 70.46%


('DEFECT', 70.46042084693909)

In [60]:
predict_single_image(
    NORMAL_DIR / "T1-D0-001.jpg"
)

Prediction: NORMAL
Confidence: 76.03%


('NORMAL', 76.02578401565552)

In [61]:
model.save(MODEL_PATH)

print("Model saved:")
print(MODEL_PATH)

Model saved:
D:\proto\TextileDefectPrototype\models\mobilenetv2_classifier.keras


In [62]:
predict_single_image(
    NORMAL_DIR / "T1-D0-001.jpg"
)

Prediction: NORMAL
Confidence: 76.03%


('NORMAL', 76.02578401565552)

In [63]:
predict_single_image(
    DEFECT_DIR / "T1-D2-004.jpg"
)

Prediction: DEFECT
Confidence: 70.46%


('DEFECT', 70.46042084693909)

In [64]:
print(MODEL_PATH)
print(MODEL_PATH.exists())

D:\proto\TextileDefectPrototype\models\mobilenetv2_classifier.keras
True


In [65]:
model.save(MODEL_PATH)
print("Saved:", MODEL_PATH)

Saved: D:\proto\TextileDefectPrototype\models\mobilenetv2_classifier.keras


In [66]:
import tensorflow as tf
import numpy as np

saved_model = tf.keras.models.load_model(MODEL_PATH)

print("Saved model loaded successfully.")
print("Model path:", MODEL_PATH)

Saved model loaded successfully.
Model path: D:\proto\TextileDefectPrototype\models\mobilenetv2_classifier.keras


In [67]:
def test_saved_model(image_path):
    image = tf.io.read_file(str(image_path))

    image = tf.image.decode_image(
        image,
        channels=3,
        expand_animations=False
    )

    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32)
    image = preprocess_input(image)
    image = tf.expand_dims(image, axis=0)

    raw_prediction = float(
        saved_model.predict(image, verbose=0)[0][0]
    )

    if raw_prediction >= 0.5:
        predicted_class = "DEFECT"
        confidence = raw_prediction * 100
    else:
        predicted_class = "NORMAL"
        confidence = (1 - raw_prediction) * 100

    print("Raw prediction:", raw_prediction)
    print("Prediction:", predicted_class)
    print("Confidence:", f"{confidence:.2f}%")

    return predicted_class, confidence

In [68]:
test_saved_model(
    DEFECT_DIR / "T1-D2-004.jpg"
)

Raw prediction: 0.7046042084693909
Prediction: DEFECT
Confidence: 70.46%


('DEFECT', 70.46042084693909)